In [1]:
# imports
import pandas as pd
import numpy as np
import xgboost as xgb
from xgboost import XGBClassifier
import shap
import sklearn

c:\Users\will6\miniconda3\envs\cs320\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
#### BASE

#tested various hyperparameters

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


training_dfs = []
base_val_dfs = []
val_years = []

years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    base_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

base_val_error = []

#model
base_mod = XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 12, max_bin = 20, min_child_weight = 2, num_parallel_tree = 1, objective='binary:logistic', seed = 323)

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = base_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    base_mod.fit(X_train, y_train)
    
    #make predictions
    preds = base_mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    base_val_dfs[i] = val.copy()
    print(val_years[i], "Loss:", np.mean(val['loss']))
    base_val_error.append(np.mean(val['loss']))

print(np.mean(base_val_error[-5:]))

#print(val_base.drop('loss', axis = 1).sort_values('pred', ascending = False).head(5))
#val_base.drop('loss', axis = 1).sort_values('pred', ascending = True).head(5)

#shap
explainer = shap.TreeExplainer(base_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1974814012860625
2013 Loss: 0.20004218705746035
2014 Loss: 0.20425875563683518
2015 Loss: 0.17312351795592454
2016 Loss: 0.19560346521596406
2017 Loss: 0.1835725788662484
2018 Loss: 0.202243712122715
2019 Loss: 0.17347296459017142
2021 Loss: 0.22116188201106857
2022 Loss: 0.22264053183889349
2023 Loss: 0.1986697337199703
2024 Loss: 0.19643910984265714
2025 Loss: 0.15934492792347577
0.19965123706721305


In [3]:
### Incremental Learning

#Data

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

training_dfs = []
il_val_dfs = []
val_years = []

first_year = 2003
training = training.query("Season >= @first_year").copy()

cutoff_year = 2019

years = [y for y in range(cutoff_year, 2025) if y != 2020]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    seasons = [i]

    if i == cutoff_year:
        temp_training = training.query("Season <= @i")
    else:
        temp_training = training.query("Season in @seasons")
    
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    il_val_dfs.append(temp_val)
    val_years.append(val_year)

    # Model

#model
il_mod = XGBClassifier(n_estimators=400, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, max_bin = 20, min_split_loss = 15, min_child_weight = 2, objective='binary:logistic', seed=323)

il_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = il_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    if(i == 0):
        il_mod.fit(X_train, y_train)
    else:
        il_mod.set_params(n_estimators=il_mod.get_booster().num_boosted_rounds() + 50, max_depth = 3, learning_rate = 0.03)
        il_mod.fit(X_train, y_train, xgb_model=il_mod.get_booster()
    )
    
#round(mod.get_booster().num_boosted_rounds()*0.08)

    #make predictions
    preds = il_mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    il_val_dfs[i] = val.copy()
    il_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])


print("Last 5 Loss:", np.mean(il_val_error[-5:]), " | Dif:", np.mean(il_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(il_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)

2021 Loss: 0.22152399183042915  |  Dif: 0.02592052661446509
2022 Loss: 0.22184573596660553  |  Dif: 0.03827315710035714
2023 Loss: 0.20275984988765444  |  Dif: 0.000516137764939445
2024 Loss: 0.1981018661123207  |  Dif: 0.024628901522149277
2025 Loss: 0.16061264233966124  |  Dif: -0.06054923967140732
Last 5 Loss: 0.20096881722733423  | Dif: 0.001317580160121179


In [4]:
#### Season as Parameter

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['Season',
            'seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


training_dfs = []
s_val_dfs = []
val_years = []

years = [2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    s_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
s_mod = XGBClassifier(n_estimators=600, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 12, max_bin = 20, min_child_weight = 2, objective='binary:logistic', seed = 323)

s_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = s_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    s_mod.fit(X_train, y_train)
    
    #make predictions
    preds = s_mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    s_val_dfs[i] = val.copy()
    s_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])


print("Last 5 Loss:", np.mean(s_val_error[-5:]), " | Dif:", np.mean(s_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(s_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2016 Loss: 0.19532169132289753  |  Dif: 0.03597676339942177
2017 Loss: 0.18283335287613472  |  Dif: -0.014648048409927783
2018 Loss: 0.20229216086061685  |  Dif: 0.002249973803156502
2019 Loss: 0.1735392927531772  |  Dif: -0.030719462883657983
2021 Loss: 0.22123557599426438  |  Dif: 0.025632110778300327
2022 Loss: 0.2231876104948596  |  Dif: 0.039615031628611214
2023 Loss: 0.19949853816667018  |  Dif: -0.0027451739560448163
2024 Loss: 0.19649935718082956  |  Dif: 0.02302639259065814
2025 Loss: 0.1591221886426459  |  Dif: -0.06203969336842266
Last 5 Loss: 0.19990865409585395  | Dif: 0.000257417028640905


In [5]:
#Rolling

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            #'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            #'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]


training_dfs = []
r_val_dfs = []
val_years = []

y_count = 8

years = [2010,2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    if i <= 2019 or i >= 2020 + y_count:
        seasons = [i - k for k in range(8)]
    else:
        seasons = [i - k for k in range(9)]

    temp_training = training.query("Season in @seasons")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    r_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
r_mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 4, max_bin = 20, min_child_weight = 4, objective='binary:logistic', seed = 323)

r_val_error = []

for i in range(len(years)):
    X_train = training_dfs[i][features]
    y_train = training_dfs[i]['result']

    val = r_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    r_mod.fit(X_train, y_train)
    
    #make predictions
    preds = r_mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    r_val_dfs[i] = val.copy()
    r_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(r_val_error[-5:]), " | Dif:", np.mean(r_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(r_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2011 Loss: 0.22553568490739476  |  Dif: 0.05206272031722334
2012 Loss: 0.19950623903519016  |  Dif: -0.02165564297587841
2013 Loss: 0.19490476597835282  |  Dif: -0.027735765860540662
2014 Loss: 0.20900683072203602  |  Dif: 0.010337097002065726
2015 Loss: 0.1722514512334501  |  Dif: -0.024187658609207047
2016 Loss: 0.20399947568629492  |  Dif: 0.04465454776281916
2017 Loss: 0.1807728205447766  |  Dif: -0.016708580741285894
2018 Loss: 0.2085208569208964  |  Dif: 0.008478669863436039
2019 Loss: 0.17343607541471237  |  Dif: -0.03082268022212281
2021 Loss: 0.22514173147557184  |  Dif: 0.029538266259607787
2022 Loss: 0.22547828356486055  |  Dif: 0.041905704698612156
2023 Loss: 0.19517543897910183  |  Dif: -0.007068273143613163
2024 Loss: 0.1973936539310386  |  Dif: 0.02392068934086719
2025 Loss: 0.15429998107163542  |  Dif: -0.06686190093943314
Last 5 Loss: 0.19949781780444165  | Dif: -0.00015341926277140372


In [12]:
#### Weights

#read csv
training = pd.read_csv("train_data_m.csv")

#features
features = ['seed_dif', 
            'massey_rank_dif', 
            'avg_margin_dif',
            'avg_eff_A',
            'avg_opp_eff_A', 
            'avg_eff_B', 
            'avg_opp_eff_B',
            'avg_thr_per_A', 
            'ft_per_A', 
            'avg_fg_per_A', 
            'avg_fg_a_per_A',
            'avg_thr_a_per_A', 
            'avg_to_per_A', 
            'avg_blk_per_A',
            'avg_opp_fg_a_per_A', 
            'avg_opp_fg_per_A', 
            'avg_opp_to_per_A',
            'avg_or_per_A',
            'avg_foul_rec_per_A', 
            'avg_foul_per_A',
            'avg_thr_per_B', 
            'ft_per_B', 
            'avg_fg_per_B', 
            'avg_fg_a_per_B',
            'avg_thr_a_per_B', 
            'avg_to_per_B', 
            'avg_blk_per_B',
            'avg_opp_fg_a_per_B', 
            'avg_opp_fg_per_B', 
            'avg_opp_to_per_B',
            'avg_or_per_B',
            'avg_foul_rec_per_B', 
            'avg_foul_per_B',
            'thr_a_per_dif', 
            'tempo_pred', 
            'pred_fg_per_dif', 
            'pred_or_per_dif'
            ]

training_dfs = []
w_val_dfs = []
val_years = []

weight_param = 0.95

years = [2011,2012,2013,2014,2015,2016,2017,2018,2019,2021,2022,2023,2024]

for i in years:
    if i == 2019:
        val_year = 2021
    else:
        val_year = i+1

    temp_training = training.query("Season <= @i")
    temp_val = training.query("Season == @val_year")

    training_dfs.append(temp_training)
    w_val_dfs.append(temp_val)
    val_years.append(val_year)

### Model

#model
w_mod = XGBClassifier(n_estimators=500, max_depth=3, learning_rate=0.01, subsample = 0.6, colsample_bynode = 0.8, min_split_loss = 6, max_bin = 20, min_child_weight = 2, objective='binary:logistic', seed = 323)

w_val_error = []

for i in range(len(years)):
    temp_df = training_dfs[i]
    temp_df = temp_df.assign(weight = (weight_param ** (temp_df['Season'].max() - temp_df['Season'])).clip(0.1))

    X_train = temp_df[features]
    y_train = temp_df['result']
    train_weights = temp_df['weight']

    val = w_val_dfs[i].copy()

    X_val = val[features]
    y_val = val['result']

    #fit model
    w_mod.fit(X_train, y_train, sample_weight=train_weights)
    
    #make predictions
    preds = w_mod.predict_proba(X_val)

    val['pred'] = preds[:,1]

    val['loss'] = (val['pred'] - val['result'])**2

    w_val_dfs[i] = val.copy()
    w_val_error.append(np.mean(val['loss']))
    print(val_years[i], "Loss:", np.mean(val['loss']), " |  Dif:", np.mean(val['loss']) - base_val_error[val_years[i] - 2017])

print("Last 5 Loss:", np.mean(w_val_error[-5:]), " | Dif:", np.mean(w_val_error[-5:]) - np.mean(base_val_error[-5:]))

#shap
explainer = shap.TreeExplainer(w_mod)
shap_values = explainer.shap_values(X_val)
#shap.summary_plot(shap_values, X_val, plot_type="bar", max_display=50)


2012 Loss: 0.1993957849423377  |  Dif: -0.021766097068730866
2013 Loss: 0.1976038274781247  |  Dif: -0.025036704360768797
2014 Loss: 0.20519535744443526  |  Dif: 0.006525623724464968
2015 Loss: 0.17459440854376498  |  Dif: -0.021844701298892155
2016 Loss: 0.19874537580837823  |  Dif: 0.03940044788490246
2017 Loss: 0.18160378505596517  |  Dif: -0.01587761623009734
2018 Loss: 0.20339080020509484  |  Dif: 0.0033486131476344883
2019 Loss: 0.1712323797296404  |  Dif: -0.03302637590719479
2021 Loss: 0.21828364411018925  |  Dif: 0.02268017889422519
2022 Loss: 0.2218742287991676  |  Dif: 0.03830164993291921
2023 Loss: 0.1985804930145373  |  Dif: -0.0036632191081776844
2024 Loss: 0.19768785612013948  |  Dif: 0.024214891529968058
2025 Loss: 0.1595193215233308  |  Dif: -0.06164256048773778
Last 5 Loss: 0.1991891087134729  | Dif: -0.0004621283537401544


In [34]:
### Combine

#Merge Data
def get_vals(lst, prefix):
    df = pd.concat(lst, ignore_index=True)[['Season', 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result', 'pred', 'loss']]
    df = df.rename({'pred': f'{prefix}pred', 'loss': f'{prefix}loss'}, axis='columns')
    return df

base_val = get_vals(base_val_dfs, "base_")
il_val = get_vals(il_val_dfs, "il_")
s_val = get_vals(s_val_dfs, "s_")
r_val = get_vals(r_val_dfs, "r_")
w_val = get_vals(w_val_dfs, "w_")

merge_keys = ["Season", 'DayNum', 'team_A', 'team_B', 'score_A', 'score_B', 'result']

full_val = pd.merge(base_val, il_val, how="outer", on=merge_keys).merge(
        s_val, how="outer", on=merge_keys).merge(
        r_val, how="outer", on=merge_keys).merge(
        w_val, how="outer", on=merge_keys)


full_val.tail()


#combined prediction
base_weight = 0.4
il_weight = 0
s_weight = 0
r_weight = 0.3
w_weight = 0.3

full_val = full_val.assign(com_pred = full_val['base_pred']*base_weight + full_val['il_pred']*il_weight + full_val['s_pred']*s_weight + full_val['r_pred']*r_weight + full_val['w_pred']*w_weight)
full_val['com_loss'] = (full_val['com_pred'] - full_val['result'])**2

full_val.tail()

#group and summarize
val_summary = full_val.groupby(["Season"]).agg(
    base_val=("base_loss", "mean"),
    il_val=("il_loss", "mean"),
    s_val=("s_loss", "mean"),
    r_val=("r_loss", "mean"),
    w_val=("w_loss", "mean"),
    com_val=("com_loss", "mean")).reset_index()

val_summary.loc['last5'] = val_summary.set_index('Season').iloc[-5:].mean()

val_summary.tail(6)

,Season,base_val,il_val,s_val,r_val,w_val,com_val
9,2021.0,0.221162,0.221524,0.221236,0.225142,0.218284,0.220860
10,2022.0,0.222641,0.221846,0.223188,0.225478,0.221874,0.222488
11,2023.0,0.198670,0.202760,0.199499,0.195175,0.198580,0.196742
12,2024.0,0.196439,0.198102,0.196499,0.197394,0.197688,0.196661
13,2025.0,0.159345,0.160613,0.159122,0.154300,0.159519,0.157388
last5,NaN,0.199651,0.200969,0.199909,0.199498,0.199189,0.198828
